# County Rainfall and Planting Advisory

A farmer-support NGO sends SMS advice on planting windows in five counties. The NGO holds monthly rainfall readings from local weather stations. One station typed its readings by hand and the data holds errors.

The notebook cleans that data and analyses rainfall with NumPy. It ends with a short planting advisory and a note on data quality. The results save to `rainfall_summary.csv`.

## Setup

**Approach:** NumPy handles every calculation. The built-in `csv` module writes the summary file. The list `MONTHS` turns a month position into a month name.

In [1]:
import csv
import numpy as np

MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

## Dataset

Rows are counties and columns are months from January to December. Values are rainfall in mm. The Machakos row arrived as raw text and needs cleaning first.

In [2]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## Task 1: Cleaning the Machakos readings

**Approach:** The function `clean_station_row()` tries to convert each reading with `int()`. A reading such as N/A or an empty string raises a `ValueError` and the function catches that specific error. The function then averages the valid readings and rounds the result. Every invalid month receives that value. The function also reports which months it imputed.

In [3]:
def clean_station_row(raw_values, month_names):
    """Convert raw text readings to integers and fill invalid ones.

    Valid readings become integers. Invalid readings are replaced with the
    rounded average of the valid readings in the same row.

    Args:
        raw_values: a list of 12 readings typed as text.
        month_names: a list of 12 month names in the same order.

    Returns:
        A tuple (cleaned, imputed_months, fill_value) where cleaned is a list
        of integers, imputed_months lists the months that were filled and
        fill_value is the rounded average used for them.
    """
    converted = []
    for value in raw_values:
        try:
            number = int(value)
            if number < 0:
                raise ValueError("rainfall cannot be negative")
            converted.append(number)
        except ValueError:
            converted.append(None)

    valid = [number for number in converted if number is not None]
    fill_value = round(sum(valid) / len(valid))

    cleaned = []
    imputed_months = []
    for month, number in zip(month_names, converted):
        if number is None:
            cleaned.append(fill_value)
            imputed_months.append(month)
        else:
            cleaned.append(number)

    return cleaned, imputed_months, fill_value


machakos_clean, imputed_months, fill_value = clean_station_row(machakos_raw, MONTHS)

print("Imputed months:", ", ".join(imputed_months))
print(f"Value used for each imputed month: {fill_value} mm")
print("Cleaned Machakos row:", machakos_clean)

Imputed months: April, September
Value used for each imputed month: 68 mm
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Task 2: One 5 by 12 array

**Approach:** `np.array()` stacks the four clean rows and the cleaned Machakos row into one array. Rows are counties and columns are months. Printing `shape` and `dtype` confirms the layout.

In [4]:
rainfall = np.array(clean_rows + [machakos_clean], dtype=np.int64)

print("Shape:", rainfall.shape)
print("dtype:", rainfall.dtype)

Shape: (5, 12)
dtype: int64


## Task 3: Annual totals and monthly averages

**Approach:** `sum(axis=1)` adds across the months of each county. `mean(axis=0)` averages down each month column across all five counties. NumPy performs both calculations and no loop is needed. A short loop only prints the results.

In [5]:
annual_totals = rainfall.sum(axis=1)
monthly_averages = rainfall.mean(axis=0)
overall_average = rainfall.mean()

print("Annual rainfall per county (mm)")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<10}{total:>7,}")

print("\nAverage rainfall per month across all counties (mm)")
for month, average in zip(MONTHS, monthly_averages):
    print(f"  {month:<10}{average:>7.1f}")

print(f"\nAverage across all months and counties: {overall_average:.1f} mm")

Annual rainfall per county (mm)
  Nairobi     1,045
  Nakuru        916
  Kisumu      1,371
  Mombasa     1,075
  Machakos      811

Average rainfall per month across all counties (mm)
  January      47.0
  February     47.6
  March        91.0
  April       164.4
  May         171.2
  June         67.6
  July         56.6
  August       58.2
  September    64.2
  October      67.8
  November    127.4
  December     80.6

Average across all months and counties: 87.0 mm


## Task 4: Wettest month per county

**Approach:** `np.argmax(rainfall, axis=1)` returns the column position of the highest reading in each row. Indexing the `MONTHS` array with those positions gives month names. A `zip()` loop pairs each county with its result for printing.

In [6]:
wettest_idx = np.argmax(rainfall, axis=1)
wettest_months = np.array(MONTHS)[wettest_idx]
wettest_mm = rainfall.max(axis=1)

print("Wettest month per county")
for county, month, mm in zip(counties, wettest_months, wettest_mm):
    print(f"  {county:<10}{month:<10}{mm:>5} mm")

Wettest month per county
  Nairobi   April       219 mm
  Nakuru    April       145 mm
  Kisumu    April       240 mm
  Mombasa   May         240 mm
  Machakos  November    182 mm


## Task 5: Dry months

**Approach:** The comparison `rainfall < 50` builds a True and False mask with the same shape as the data. Each True counts as 1 so `sum(axis=1)` counts the dry months in each county.

In [7]:
dry_mask = rainfall < 50
dry_counts = dry_mask.sum(axis=1)

print("Dry months per county (below 50 mm)")
for county, count in zip(counties, dry_counts):
    print(f"  {county:<10}{count}")

Dry months per county (below 50 mm)
  Nairobi   4
  Nakuru    3
  Kisumu    0
  Mombasa   2
  Machakos  6


## Task 6: Planting windows

**Approach:** The shifted array technique adds each month to the next month with `rainfall[:, :-1] + rainfall[:, 1:]`. Column 0 of the result holds January plus February and so on. `np.argmax(..., axis=1)` finds the strongest pair in each row. That pair is the recommended planting window.

In [8]:
pair_totals = rainfall[:, :-1] + rainfall[:, 1:]
best_start = np.argmax(pair_totals, axis=1)
best_pair_mm = pair_totals.max(axis=1)

month_names = np.array(MONTHS)
planting_windows = np.char.add(
    np.char.add(month_names[best_start], " to "),
    month_names[best_start + 1],
)

print("Planting window per county (highest two-month rainfall)")
for county, window, mm in zip(counties, planting_windows, best_pair_mm):
    print(f"  {county:<10}{window:<22}{mm:>5} mm")

Planting window per county (highest two-month rainfall)
  Nairobi   April to May            395 mm
  Nakuru    April to May            265 mm
  Kisumu    April to May            420 mm
  Mombasa   April to May            390 mm
  Machakos  November to December    278 mm


## Task 7: Writing the summary file

**Approach:** The `csv` module writes one header row and one row for each county. A second block reads the file back to confirm the content.

In [9]:
with open("rainfall_summary.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["county", "annual_total_mm", "wettest_month",
                     "dry_months", "planting_window"])
    for county, total, month, dry, window in zip(
        counties, annual_totals, wettest_months, dry_counts, planting_windows
    ):
        writer.writerow([county, int(total), str(month), int(dry), str(window)])

with open("rainfall_summary.csv", newline="") as file:
    for row in csv.reader(file):
        print(row)

['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
['Nairobi', '1045', 'April', '4', 'April to May']
['Nakuru', '916', 'April', '3', 'April to May']
['Kisumu', '1371', 'April', '0', 'April to May']
['Mombasa', '1075', 'May', '2', 'April to May']
['Machakos', '811', 'November', '6', 'November to December']


## Task 8: Advisory for the NGO

**Approach:** The advisory uses figures from Tasks 1 to 7. It gives one planting window for each group of counties and one warning about dry months. The last sentence carries the data-quality caveat. The Data Quality Notes section follows with the details.

## Planting Advisory

Farmers in Nairobi and Nakuru and Kisumu and Mombasa should plant in April and May. That window brings the most rain in all four counties with 420 mm in Kisumu and 395 mm in Nairobi and 390 mm in Mombasa and 265 mm in Nakuru. Machakos farmers should wait for November and December when the county receives 278 mm in two months. Machakos also has 6 dry months below 50 mm so farmers there need drought-tolerant seed and water storage. Kisumu carries the lowest rain risk with 1,371 mm a year and no dry month. Machakos readings for April and September were invalid and replaced with the county average of 68 mm so the Machakos annual total of 811 mm includes 136 mm of estimates and needs checking against the station records.

## Data Quality Notes

- The Machakos row arrived as raw text with two invalid entries. April held N/A and September held an empty string.
- One entry held extra spaces around the number. Python `int()` strips spaces so the entry converted to 20 with no repair.
- The two invalid months received 68 mm. That value is the rounded average of the ten valid readings (67.5 mm).
- Imputed values hide real variation. The Machakos annual total of 811 mm includes 136 mm of estimates and is the least reliable figure in the analysis.
- The dry month count for Machakos stays the same under imputation because 68 mm sits above the 50 mm threshold. The recommended window rests on real November and December readings.
- The data covers one year for each county. A planting window from a single year can differ from the long-term pattern so a multi-year check would strengthen the advice.